In [ ]:
!pip install opencv-python-headless dlib matplotlib numpy -q
!wget -q https://github.com/davisking/dlib-models/raw/master/mmod_human_face_detector.dat.bz2
!bzip2 -d mmod_human_face_detector.dat.bz2
import os
os.makedirs('Weights', exist_ok=True)
!mv mmod_human_face_detector.dat Weights/
print('Setup Done!')

In [ ]:
import cv2
import dlib
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

In [ ]:
def take_photo():
    js = Javascript('''
        async function takePhoto() {
            const div = document.createElement('div');
            div.style.cssText = 'position:fixed;top:0;left:0;width:100%;height:100%;background:rgba(0,0,0,0.8);z-index:9999;display:flex;flex-direction:column;align-items:center;justify-content:center;';
            
            const video = document.createElement('video');
            video.style.cssText = 'width:640px;height:480px;border-radius:10px;';
            
            const btn = document.createElement('button');
            btn.textContent = '📸 Photo Lo';
            btn.style.cssText = 'margin-top:20px;padding:15px 40px;font-size:20px;background:#00c853;color:white;border:none;border-radius:30px;cursor:pointer;';
            
            div.appendChild(video);
            div.appendChild(btn);
            document.body.appendChild(div);
            
            const stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = stream;
            await video.play();
            
            await new Promise((resolve) => btn.onclick = resolve);
            
            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getTracks().forEach(track => track.stop());
            div.remove();
            return canvas.toDataURL('image/jpeg', 0.8);
        }
        takePhoto()
    ''')
    display(js)
    data = eval_js('takePhoto()')
    binary = b64decode(data.split(',')[1])
    image = np.frombuffer(binary, dtype=np.uint8)
    image = cv2.imdecode(image, cv2.IMREAD_COLOR)
    return image

In [ ]:
print('📸 Camera khulega - Photo Lo button dabao!')
image = take_photo()
print('Image Ready!')

In [ ]:
image = cv2.resize(image, (800, 600))
image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 7))
plt.imshow(img_rgb)
plt.title('Original Image')
plt.axis('off')
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
plt.imshow(image_gray, cmap='gray')
plt.title('Grayscale Image')
plt.axis('off')
plt.show()

In [ ]:
face_cascade_path = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
face_detector = cv2.CascadeClassifier(face_cascade_path)

haar_image = image.copy()
detections = face_detector.detectMultiScale(image_gray)
print('Haar Cascade - Faces found:', len(detections))

for (x, y, w, h) in detections:
    cv2.rectangle(haar_image, (x, y), (x+w, y+h), (0, 255, 0), 2)

img_rgb = cv2.cvtColor(haar_image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 7))
plt.imshow(img_rgb)
plt.title(f'Haar Cascade - Faces Detected: {len(detections)}')
plt.axis('off')
plt.show()

In [ ]:
face_detector_hog = dlib.get_frontal_face_detector()

hog_image = image.copy()
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
detections_hog = face_detector_hog(image_rgb, 1)
print('HOG Detector - Faces found:', len(detections_hog))

for face in detections_hog:
    l, t, r, b = face.left(), face.top(), face.right(), face.bottom()
    cv2.rectangle(hog_image, (l, t), (r, b), (0, 255, 255), 2)

img_rgb = cv2.cvtColor(hog_image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 7))
plt.imshow(img_rgb)
plt.title(f'HOG Detector - Faces Detected: {len(detections_hog)}')
plt.axis('off')
plt.show()

In [ ]:
cnn_detector = dlib.cnn_face_detection_model_v1('Weights/mmod_human_face_detector.dat')

cnn_image = image.copy()
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
detections_cnn = cnn_detector(image_rgb, 1)
print('CNN Detector - Faces found:', len(detections_cnn))

for face in detections_cnn:
    l = face.rect.left()
    t = face.rect.top()
    r = face.rect.right()
    b = face.rect.bottom()
    cv2.rectangle(cnn_image, (l, t), (r, b), (255, 255, 0), 2)

img_rgb = cv2.cvtColor(cnn_image, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 7))
plt.imshow(img_rgb)
plt.title(f'CNN Detector - Faces Detected: {len(detections_cnn)}')
plt.axis('off')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

haar_rgb = cv2.cvtColor(haar_image, cv2.COLOR_BGR2RGB)
hog_rgb = cv2.cvtColor(hog_image, cv2.COLOR_BGR2RGB)
cnn_rgb = cv2.cvtColor(cnn_image, cv2.COLOR_BGR2RGB)

axes[0].imshow(haar_rgb)
axes[0].set_title(f'Haar Cascade\nFaces: {len(detections)}')
axes[0].axis('off')

axes[1].imshow(hog_rgb)
axes[1].set_title(f'HOG Detector\nFaces: {len(detections_hog)}')
axes[1].axis('off')

axes[2].imshow(cnn_rgb)
axes[2].set_title(f'CNN Detector\nFaces: {len(detections_cnn)}')
axes[2].axis('off')

plt.suptitle('Face Detection - Method Comparison', fontsize=16)
plt.tight_layout()
plt.show()

print('Haar Cascade Faces:', len(detections))
print('HOG Detector Faces:', len(detections_hog))
print('CNN Detector Faces:', len(detections_cnn))